In [1]:
import json
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/seminar/

/content/drive/MyDrive/seminar


In [4]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch
import os
from IPython.display import display

# Model ID
model_id = "Eren-Senoglu/llava-med-v1.5-mistral-7b-hf"

# Load processor and model
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Optional but important: set patch size if missing
if hasattr(processor.image_processor, "patch_size") and processor.image_processor.patch_size is None:
    processor.image_processor.patch_size = 14
else:
    processor.patch_size = 14

# Load image
#image_path = "/content/drive/MyDrive/seminar/0cbaca12e05803e6301f8f4a92b47565.png"
images_path ="/content/drive/MyDrive/seminar/images"
results = []



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

model-00004-of-00004.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

In [28]:
counter=0
for image_name in os.listdir(images_path):
  counter+=1
  print(counter)
  image_path = os.path.join(images_path, image_name)
  image = Image.open(image_path).convert("RGB")

  # visualize the image
  #display(image)

  # unhealthy bias
  #prompt = """USER: <image>\nCarefully review and analyze the chest x-ray. Classify it as healthy or unhealthy.\nASSISTANT:"""

  # healthy bias
  '''prompt = (
    "USER: <image>\n"
    "You are a radiologist tasked with binary classification of a chest X-ray. "
    "Classify the image as either 'Healthy' or 'Unhealthy' based on visual findings. "
    "Do not assume the image is healthy or unhealthy — base your decision solely on what you see. "
    "Respond with only one word.\n"
    "ASSISTANT:"
)'''
  # this one seems reasonable
  prompt = (
    "USER: <image>\n"
    "You are an expert radiologist. Examine the image. "
    "Healthy or Unhealthy? "
    "Respond with only one word.\n"
    "ASSISTANT:"
)
  '''# some variety but leaning unhealthy...
  prompt = (
    "USER: <image>\n"
    "Healthy or Unhealthy? "
    "Respond with only one word.\n"
    "ASSISTANT:"'''


  # Prepare inputs (text prompt + image)
  inputs = processor(
      text=prompt,
      images=image,
      return_tensors="pt"
  )

  #print("inputs:", inputs)
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  # Run inference
  with torch.no_grad():
      generated_ids = model.generate(
          **inputs,
          max_new_tokens=12,
          min_new_tokens=1,
          do_sample=False,
          pad_token_id=processor.tokenizer.pad_token_id,
          eos_token_id=processor.tokenizer.eos_token_id,
          use_cache=True
      )

  # Decode result
  full_response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

  # Extract only the assistant's part
  if "ASSISTANT:" in full_response:
      assistant_response = full_response.split("ASSISTANT:")[-1].strip()
  else:
      assistant_response = full_response

  #print("Full response:", full_response)
  print("Assistant response:", assistant_response)

  results.append({
            'image': image_name,
            'prediction': assistant_response
        })

1
Assistant response: The chest X-ray appears to be unhealthy.
2
Assistant response: The chest X-ray appears to be unhealthy.
3
Assistant response: Healthy.
4
Assistant response: Healthy.
5
Assistant response: Healthy.
6
Assistant response: Healthy.
7
Assistant response: Healthy.
8
Assistant response: Healthy.
9
Assistant response: Healthy.
10
Assistant response: The image shows a healthy chest X-ray.
11
Assistant response: The chest X-ray appears to be unhealthy.
12
Assistant response: Healthy.
13
Assistant response: The chest X-ray appears to be unhealthy.
14
Assistant response: Healthy.
15
Assistant response: Healthy.
16
Assistant response: The chest X-ray appears to be unhealthy.
17
Assistant response: The chest X-ray appears to be unhealthy.
18
Assistant response: The chest X-ray appears to be unhealthy.
19
Assistant response: Healthy.
20
Assistant response: Healthy.
21
Assistant response: Healthy.
22
Assistant response: Healthy.
23
Assistant response: Healthy.
24
Assistant respon

In [ ]:
import json
with open('annotations_len_50.json', 'r') as f:
    annotations = json.load(f)

for image_name in os.listdir(images_path):
  disease_list = []
  #strip the .png
  id = image_name.split('.')[0]
  if id not in annotations:
      print(f"Warning: Annotation for image ID {id} not found.")
  else:
      print(f"Processing x-ray image {id}")
      if annotations[id]['bbox_2d']:
          for entry in annotations[id]['bbox_2d']:
              disease = entry[4]
              disease_list.append(disease)
          image_path = os.path.join(images_path, image_name)
          image = Image.open(image_path).convert("RGB")
          for disease in disease_list:
              prompt = f"""USER: <image>\n Carefully analyze the provided chest X-ray and locate the {disease}. Provide ONLY the bounding box coordinates for the {disease} in the following format: (x1, y1, x2, y2). ASSISTANT:"""
              inputs = processor(
                  text=prompt,
                  images=image,
                  return_tensors="pt"
              )

              #print("inputs:", inputs)
              inputs = {k: v.to(model.device) for k, v in inputs.items()}

              # Run inference
              with torch.no_grad():
                  generated_ids = model.generate(
                      **inputs,
                      max_new_tokens=50, # Increased max_new_tokens to allow for full coordinate output
                      min_new_tokens=1,
                      do_sample=False,
                      temperature=0.7,
                      top_p = 0.8,
                      pad_token_id=processor.tokenizer.pad_token_id,
                      eos_token_id=processor.tokenizer.eos_token_id,
                      use_cache=True
                  )

              # Decode result
              full_response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

              # Extract only the assistant's part
              if "ASSISTANT:" in full_response:
                  assistant_response = full_response.split("ASSISTANT:")[-1].strip()
              else:
                  assistant_response = full_response

              #print("Full response:", full_response)
              print("Assistant response:", assistant_response)

              results.append({
                        'image': image_name,
                        'prediction': assistant_response
                    })

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Processing x-ray image c4b7716467a0e8cf4f76c68dbca0a3bd
Processing x-ray image e5dbb0754f79f54cf9c052355d63dfff
Processing x-ray image 0570e5d8e3e4532ca90597a979d1e7ff
Processing x-ray image face7d0feea230cb057697e34fcb3ad3
Processing x-ray image 0cc5600e54fe599fefce87de375268e3
Processing x-ray image 5562ea946b0ed8574dd20d05a001d6c4


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Assistant response: The Lung Opacity is located at (300, 400, 400, 500).


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

Assistant response: The bounding box coordinates for the Rib fracture are (100, 100, 150, 150).
Assistant response: The bounding box coordinates for the ILD in the chest X-ray are (100, 100, 150, 150).
Processing x-ray image 5a5364fb4f700d6a3daad03dc7a3eefc
Processing x-ray image 2cd35cf79e8d14f189f56948098c27ee
Processing x-ray image cf1725e48bb0358f018494268da8dc1a
Processing x-ray image 0cbaca12e05803e6301f8f4a92b47565
Processing x-ray image aee0d37d1129e6666c5e6158c400d5a7
Processing x-ray image 7e8a3b0d2ab003b958a70264fccb2d43
Processing x-ray image 8de556d9cd8d026b8eba03870cc6acba


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Assistant response: The bounding box coordinates for the Pleural effusion are (100, 100, 100, 100).


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Assistant response: The bounding box coordinates for the Pulmonary fibrosis in the chest X-ray are (100, 100, 150, 150).
Assistant response: The Lung Opacity is located at (300, 400, 400, 500).
Processing x-ray image 469999b0e3acee129f96aceaa99eba0d
Processing x-ray image 9cba24255e172b86d0ad17c06c1d4e8c
Processing x-ray image 742669c2df8236b4f543fca44c166ad7
Processing x-ray image fd1fedf43a88cb1ac7b40b774eac32ca


In [29]:
import csv

#save all results to a csv file with 2 columns - image and prediction
csv_output_path = "/content/drive/MyDrive/seminar/xray_predictions_final.csv"

with open(csv_output_path, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['image', 'prediction']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # Write header
    writer.writeheader()

    # Write data
    for result in results:
        writer.writerow(result)